In [1]:
import pandas as pd
import numpy as np
import json
import os
from pandas import to_datetime

REVIEW_MOVIE_PATH = "../reviews_Movies_and_TV.json.gz"
REVIEW_BOOK_PATH = "../reviews_Books.json.gz"
SAVE_DIR = "./movie_book_cdsr_processed/"
os.makedirs(SAVE_DIR, exist_ok=True)

### 1. 数据加载

In [2]:
# 加载单个域的交互数据
def load_review_data_chunked(file_path, domain_name, chunksize=50000):
    chunks = []
    reader = pd.read_json(
        file_path,
        lines=True,
        compression='gzip',
        chunksize=chunksize
    )
    for i, chunk in enumerate(reader):
        # 只保留需要的三列，立刻释放无用列占用的内存
        chunk = chunk[['reviewerID', 'asin', 'unixReviewTime']]
        chunks.append(chunk)
        if (i + 1) % 20000 == 0:
            print(f"  {domain_name} 已处理 {len(chunks)*chunksize} 行...")
    df = pd.concat(chunks, ignore_index=True)
    # 释放 chunks 列表占用的内存
    del chunks
    df = df.rename(columns={
        'reviewerID': 'user_id',
        'asin': 'item_id',
        'unixReviewTime': 'timestamp'
    })
    df['domain'] = domain_name
    print(f"{domain_name} 域交互数据加载完成，交互数：{len(df)}")
    return df

# 加载
print("开始交互数据预处理")
movie_review = load_review_data_chunked(REVIEW_MOVIE_PATH, "movie")
book_review = load_review_data_chunked(REVIEW_BOOK_PATH, "book")
print(f"共 {len(movie_review)} 条电影记录")
print(f"共 {len(book_review)} 条图书记录")

full_review = pd.concat([movie_review, book_review], ignore_index=True)

full_review.to_json(
    os.path.join(SAVE_DIR, "full_reviews.json.gz"),
    orient="records",
    lines=True,
    compression="gzip",
    force_ascii=False
)
print(f"合并数据已保存为 json.gz，共 {len(full_review)} 条记录")

开始交互数据预处理
movie 域交互数据加载完成，交互数：4607047
book 域交互数据加载完成，交互数：22507155
共 4607047 条电影记录
共 22507155 条图书记录
合并数据已保存为 json.gz，共 27114202 条记录


In [3]:
print(full_review.head())

          user_id     item_id   timestamp domain
0  A3R5OBKS7OM2IR  0000143502  1358380800  movie
1  A3R5OBKS7OM2IR  0000143529  1380672000  movie
2   AH3QC2PC1VTGP  0000143561  1216252800  movie
3  A3LKP6WPMP9UKX  0000143588  1236902400  movie
4   AVIY68KEPQ5ZD  0000143588  1232236800  movie


In [2]:
chunk_size = 10000  # 每次读1万行
df_list = []

# 逐块读取
for chunk in pd.read_json(
    "./movie_book_cdsr_processed/full_reviews.json.gz",
    lines=True,
    compression="gzip",
    chunksize=chunk_size
):
    df_list.append(chunk)  # 把每块存起来

full_review = pd.concat(df_list, ignore_index=True)

### 2. 数据过滤

In [3]:
print("执行过滤原则") 

full_review["year"] = pd.to_datetime(full_review["timestamp"], unit="s").dt.year
full_review = full_review[full_review["year"].isin([2013, 2014])].drop(columns="year")

# 先去除物品 再去除用户
# 过滤低频物品 （item < 10）
item_interact_count = full_review.groupby("item_id").size()
valid_items = item_interact_count[item_interact_count >= 10].index
full_review = full_review[full_review["item_id"].isin(valid_items)]
print(f"过滤低频物品后：剩余交互数 {len(full_review)}，有效商品数 {len(valid_items)}")


print("\n过滤用户（双域交互+数量）")
# 规则1：仅保留双域用户（在movie和book都有交互）
user_domain_count = full_review.groupby("user_id")["domain"].nunique()
dual_users = user_domain_count[user_domain_count == 2].index
full_review = full_review[full_review["user_id"].isin(dual_users)]
print(f"保留双域用户后：交互数 {len(full_review)}，用户数 {len(dual_users)}")

# 规则2：用户总交互数 ≥ 10
user_total_interact = full_review.groupby("user_id").size()
valid_users = user_total_interact[user_total_interact >= 10].index
full_review = full_review[full_review["user_id"].isin(valid_users)]
print(f"保留总交互≥10用户后：交互数 {len(full_review)}，用户数 {len(valid_users)}")

# 规则3：用户电影、图书域的交互数 ≥ 3
user_domain_interact = full_review.groupby(["user_id", "domain"]).size().unstack(fill_value=0)
final_users = user_domain_interact[
    (user_domain_interact["movie"] >= 3) &
    (user_domain_interact["book"] >= 3)
].index
full_review = full_review[full_review["user_id"].isin(final_users)]

print(f"过滤完成：交互数 {len(full_review)}，用户数 {len(final_users)}，商品数 {len(full_review['item_id'].unique())}")

执行过滤原则
过滤低频物品后：剩余交互数 10697780，有效商品数 219250

过滤用户（双域交互+数量）
保留双域用户后：交互数 1978557，用户数 303000
保留总交互≥10用户后：交互数 996971，用户数 42165
过滤完成：交互数 531435，用户数 20738，商品数 132015


### 3. 数据集划分

In [4]:
print("数据集划分")
# 商品列表
valid_item_set = set(full_review["item_id"].unique())
with open(f"{SAVE_DIR}/valid_item_set.json", "w", encoding="utf-8") as f:
    json.dump(list(valid_item_set), f)
# 用户列表
valid_user_set = set(full_review["user_id"].unique())
with open(f"{SAVE_DIR}/valid_user_set.json", "w", encoding="utf-8") as f:
    json.dump(list(valid_user_set), f)

full_review = full_review.sort_values(by=["user_id", "timestamp"]).reset_index(drop=True)
# ID映射 
user2id = {uid: idx for idx, uid in enumerate(full_review["user_id"].unique())}
item2id = {iid: idx for idx, iid in enumerate(full_review["item_id"].unique())}
domain2id = {"movie": 0, "book": 1}

full_review["user_id"] = full_review["user_id"].map(user2id)
full_review["item_id"] = full_review["item_id"].map(item2id)
full_review["domain"] = full_review["domain"].map(domain2id)

# --- 时序划分训练/验证/测试集 ---
full_review["reverse_seq_pos"] = full_review.groupby("user_id").cumcount(ascending=False)
# 标签 0 作为 测试集
# 标签 1 作为 验证集
# 标签 >=2 作为 训练集 （采用的倒序）
train_df = full_review[full_review["reverse_seq_pos"] >= 2].drop(columns=["reverse_seq_pos"])
val_df = full_review[full_review["reverse_seq_pos"] == 1].drop(columns=["reverse_seq_pos"])
test_df = full_review[full_review["reverse_seq_pos"] == 0].drop(columns=["reverse_seq_pos"])

train_df.to_csv(f"{SAVE_DIR}/train.csv", index=False)
val_df.to_csv(f"{SAVE_DIR}/val.csv", index=False)
test_df.to_csv(f"{SAVE_DIR}/test.csv", index=False)

with open(f"{SAVE_DIR}/user2id.json", "w", encoding="utf-8") as f:
    json.dump(user2id, f, ensure_ascii=False, indent=2)
with open(f"{SAVE_DIR}/item2id.json", "w", encoding="utf-8") as f:
    json.dump(item2id, f, ensure_ascii=False, indent=2)
with open(f"{SAVE_DIR}/domain2id.json", "w", encoding="utf-8") as f:
    json.dump(domain2id, f, ensure_ascii=False, indent=2)

print(f"训练集：{len(train_df)} | 验证集：{len(val_df)} | 测试集：{len(test_df)}")

数据集划分
训练集：489959 | 验证集：20738 | 测试集：20738
